In [1]:
!fuser -k 8000/tcp || true

In [2]:
%%writefile mock_av_server.py
# -*- coding: utf-8 -*-

from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse
import json

HOST = "127.0.0.1"
PORT = 8000
VALID_API_KEY = "demo-key"


def json_bytes(payload: dict) -> bytes:
    return json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8")


class Handler(BaseHTTPRequestHandler):
    def _send_json(self, status_code: int, payload: dict) -> None:
        body = json_bytes(payload)
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _auth_ok(self) -> bool:
        api_key = self.headers.get("X-API-Key", "")
        return api_key == VALID_API_KEY

    def do_GET(self):
        parsed = urlparse(self.path)
        path = parsed.path

        if path == "/api/v1/health":
            return self._send_json(200, {"status": "ok"})

        return self._send_json(404, {"error": "not found", "path": path})

    def do_POST(self):
        parsed = urlparse(self.path)
        path = parsed.path

        if path != "/api/v1/scan":
            return self._send_json(404, {"error": "not found", "path": path})

        if not self._auth_ok():
            return self._send_json(401, {"error": "unauthorized"})

        # Читаем JSON из тела запроса
        length = int(self.headers.get("Content-Length", "0") or "0")
        raw = self.rfile.read(length) if length > 0 else b"{}"

        try:
            data = json.loads(raw.decode("utf-8"))
        except Exception:
            return self._send_json(400, {"error": "invalid json"})

        file_name = data.get("filename")
        file_hash = data.get("sha256")

        # Правило "антивируса": если хеш оканчивается на "bad" -> вредоносный, иначе чистый
        verdict = "malicious" if isinstance(file_hash, str) and file_hash.endswith("bad") else "clean"

        return self._send_json(
            200,
            {
                "verdict": verdict,
                "filename": file_name,
                "sha256": file_hash,
            },
        )

    # чтобы не засорять вывод
    def log_message(self, format, *args):
        return


def main():
    server = HTTPServer((HOST, PORT), Handler)
    print(f"Mock AV server running on http://{HOST}:{PORT}")
    print("Endpoints: GET /api/v1/health, POST /api/v1/scan")
    server.serve_forever()


if __name__ == "__main__":
    main()

Writing mock_av_server.py


In [3]:
%%writefile av_client.py
# -*- coding: utf-8 -*-

import hashlib
import os
import requests

BASE_URL = "http://127.0.0.1:8000"
API_KEY = "demo-key"


def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def main():
    # Создадим тестовый файл, если нет
    test_path = "sample.bin"
    if not os.path.exists(test_path):
        with open(test_path, "wb") as f:
            f.write(b"hello from colab\n")

    file_hash = sha256_file(test_path)

    # Проверка health
    r = requests.get(f"{BASE_URL}/api/v1/health", timeout=5)
    print("HEALTH:", r.status_code, r.json())

    # Запрос на scan
    payload = {"filename": os.path.basename(test_path), "sha256": file_hash}
    headers = {"X-API-Key": API_KEY}

    r = requests.post(f"{BASE_URL}/api/v1/scan", json=payload, headers=headers, timeout=5)
    print("SCAN:", r.status_code, r.json())


if __name__ == "__main__":
    main()

Writing av_client.py


In [4]:
!fuser -k 8000/tcp || true

In [5]:
!nohup python mock_av_server.py > server.log 2>&1 &

In [6]:
!tail -n 20 server.log


In [8]:
!python av_client.py

HEALTH: 200 {'status': 'ok'}
SCAN: 200 {'verdict': 'clean', 'filename': 'sample.bin', 'sha256': '42fe95b137c37a68da84c85ee1100ea353c7988d9459a539cfc0d7a40cd64c7c'}


In [9]:
file_hash = file_hash[:-3] + "bad"

NameError: name 'file_hash' is not defined